<a href="https://colab.research.google.com/github/AcqmalFadhilla/emotion-bert-classification/blob/master/emotion_tfx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tfx kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 kB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!chmod 600 /content/kaggle.json

In [3]:
!KAGGLE_CONFIG_DIR=/content/ kaggle datasets download -d nelgiriyewithana/emotions

 77% 12.0M/15.7M [00:00<00:00, 21.6MB/s]
100% 15.7M/15.7M [00:00<00:00, 16.8MB/s]


In [4]:
!unzip emotions.zip -d data

Archive:  emotions.zip
  inflating: data/text.csv           


In [5]:
import tensorflow as tf
import os

from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer
from tfx.proto import example_gen_pb2, trainer_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

# Set variabel

In [6]:
PIPELINE_NAME = "emotion-pipeline"
SCHEMA_PIPELINE = "emotion-tfdv-schema"

# Directory untuk menyimpan artifact yang akan dihasilkan
PIPELINE_ROOT = os.path.join("pipeline", PIPELINE_NAME)

METADATA_PATH = os.path.join("metadata", PIPELINE_NAME, 'metadata.db')

SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

In [7]:
DATA_ROOT = '/content/data'

In [8]:
interactive_content = InteractiveContext(pipeline_root=PIPELINE_ROOT)

In [9]:
import pandas as pd

data = pd.read_csv(DATA_ROOT + "/text.csv")
data.drop(columns=["Unnamed: 0"], axis=1, inplace=True)
data.head()
data.to_csv("data/text.csv", index=False)


# Data

## Data ingestion

In [10]:
outputs = example_gen_pb2.Output(
    split_config = example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name='eval', hash_buckets=2)
    ])
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=outputs)
interactive_content.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [11]:
# Get the URI of the output artifact representing the training examples, which is a directory
train_uri = os.path.join(example_gen.outputs['examples'].get()[0].uri, 'Split-train')

# Get the list of files in this directory (all compressed TFRecord files)
tfrecord_filenames = [os.path.join(train_uri, name)
                      for name in os.listdir(train_uri)]

# Create a `TFRecordDataset` to read these files
dataset = tf.data.TFRecordDataset(tfrecord_filenames, compression_type='GZIP')

# Iterate over the first record and decode it.
for tfrecord in dataset.take(1):
  serialized_example = tfrecord.numpy()
  example = tf.train.Example()
  example.ParseFromString(serialized_example)
  print(example)

features {
  feature {
    key: "label"
    value {
      int64_list {
        value: 4
      }
    }
  }
  feature {
    key: "text"
    value {
      bytes_list {
        value: "i gave up my internship with the dmrg and am feeling distraught"
      }
    }
  }
}



## Data validation

### Statistic gen

In [12]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)

interactive_content.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 2
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [13]:
interactive_content.show(statistics_gen.outputs['statistics'])

### Data schema

In [14]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"]
)

interactive_content.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 3
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [15]:
interactive_content.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'label',INT,required,,-
'text',BYTES,required,,-


### Data validator

In [16]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)

interactive_content.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 4
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [17]:
interactive_content.show(example_validator.outputs["anomalies"])

## Data preprocessing

In [18]:
!pip install tensorflow_text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 23.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 73.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.4 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.13.0
    Uninstalling tensorflow-estimator-2.13.0:
      Successfully uninstalled tensorflow-estimator-2.13.0
  Attempting uninstall: keras
    Found existing installation: keras 2.13.1
    Uninstalling keras-2.13.1:
      Successfully uninstalled keras-2.13.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.13.0
    Uninstalling tensorboard-2.13.0:
      Successfully uninstalled tensorboard-2.13.0
  Attempting uninstall: tensorflow
    Found existing install

In [19]:
TUNER_MODEL_MODULE_FILE = "emotion_model_tuner.py"

In [20]:
# tfhub_handle_preprocess = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
# bert_preprocess_model = hub.KerasLayer(tfhub_handle_preprocess)

In [23]:
%%writefile {TUNER_MODEL_MODULE_FILE}
import os
import kerastuner
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as tf_text
import tensorflow_transform as tft
from tfx_bsl.tfxio import dataset_options
from tfx.components.trainer.fn_args_utils import FnArgs
from tfx.components.tuner.component import TunerFnResult
from tfx.components.trainer.fn_args_utils import DataAccessor


tfhub_handle_preprocess = 'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3'
tfhub_handle_encoder = 'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1'

LABEL_KEY = "label"
FEATURE_KEY = "text"
SEQ_LENGTH = 64
BATCH_SIZE =32

def transformed_name(key):
  return key + "_xf"

def gzip_reader_fn(filenames):
  return tf.data.TFRecordDataset(filenames, compression_type="GZIP")

def preprocessing_fn(inputs):
  outputs = {}
  feature = tf.strings.lower(inputs[FEATURE_KEY])
  feature = tf.strings.strip(inputs[FEATURE_KEY])
  outputs[transformed_name(FEATURE_KEY)] = feature
  outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.int64)

  return outputs

def input_fn(file_pattern,
             tf_transform_output: tft.TFTransformOutput,
             batch_size: int = 100) -> tf.data.Dataset:

  transformed_feature_spec = (tf_transform_output.transformed_feature_spec().copy())

  dataset = tf.data.experimental.make_batched_features_dataset(
      file_pattern=file_pattern,
      batch_size=batch_size,
      features=transformed_feature_spec,
      reader=gzip_reader_fn,
      label_key=transformed_name(LABEL_KEY))

  return dataset.prefetch(tf.data.AUTOTUNE)

def preprocess_model_fn(seq_length=SEQ_LENGTH):

  text_input = tf.keras.layers.Input(shape=(),
                                     dtype=tf.string, name=transformed_name(FEATURE_KEY))

  preprocess = hub.load(tfhub_handle_preprocess)
  tokenizer = hub.KerasLayer(preprocess.tokenize, name="tokenizer")
  tokenized_inputs = tokenizer(text_input)

  packer = hub.KerasLayer(preprocess.bert_pack_inputs,
                          arguments={'seq_length': seq_length},
                          name="packer")
  model_inputs = packer([tokenized_inputs])
  return tf.keras.Model(text_input, model_inputs)

def get_hyperparameters() -> kerastuner.HyperParameters:
  hp = kerastuner.HyperParameters()
  hp.Choice("learning_rate", [1e-4, 1e-3, 1e-2], default=1e-3)
  hp.Choice("dropout", [0.0, 0.2, 0.3, 0.4], default=0.3)
  return hp

def model_fn(hparams: kerastuner.HyperParameters) -> tf.keras.Model:
  preprocess_model = preprocess_model_fn(SEQ_LENGTH)

  encoder = hub.KerasLayer(tfhub_handle_encoder, trainable=True)
  encoder = encoder(preprocess_model.output)
  pooled_output = encoder["pooled_output"]

  x = tf.keras.layers.Dropout(hparams.get("dropout"))(pooled_output)
  x = tf.keras.layers.Dense(6, activation="softmax")(x)

  model = tf.keras.Model(inputs=preprocess_model.input, outputs=x)
  model.compile(optimizer=tf.keras.optimizers.Adam(hparams.get("learning_rate")),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=[tf.keras.metrics.CategoricalAccuracy()])
  model.summary()

  return model


def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
  tuner = kerastuner.Hyperband(
    model_fn,
    hyperparameters=get_hyperparameters(),
    max_epochs=10,
    factor=3,
    allow_new_entries=False,
    objective="val_categorical_accuracy",
    directory=fn_args.working_dir,
    project_name="kt_randomsearch"
  )

  tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
  train_dataset = input_fn(fn_args.train_files,
                           tf_transform_output,
                           batch_size=BATCH_SIZE)

  eval_dataset = input_fn(fn_args.eval_files,
                          tf_transform_output,
                          batch_size=BATCH_SIZE)

  stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_categorical_accuracy', patience=5)
  return TunerFnResult(
      tuner=tuner,
      fit_kwargs={
          "callbacks" : [stop_early],
          'x': train_dataset,
          'validation_data': eval_dataset,
          'steps_per_epoch': fn_args.train_steps,
          'validation_steps': fn_args.eval_steps
      }
  )

def get_serve_tf_examples_fn(model, tf_transform_output):
  model.tft_layer = tf.transform_output.transform_features_layer()

  @tf.function
  def serve_tf_examples_fn(serialized_example):
    feature_spec = tf_transform_output.raw_feature_spec()
    feature_spec.pop(LABEL_KEY)
    parsed_feature = tf.io.parse_example(serialized_tf_examples, feature_spec)
    transformed_output = model.tft_layer(parsed_feature)
    return model(transformed_output)

  return serve_tf_examples_fn



def run_fn(fn_args: FnArgs):

  log_dir = os.path.join(os.path.dirname(fn_args.model_run_dir), 'logs')
  tensorboard = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    update_freq='batch'
  )
  stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_binary_accuracy', mode='max', verbose=1, patience=10)
  model_chekpoint = tf.keras.callbacks.ModelCheckpoint(fn_args.serving_model_dir, monitor='val_binary_accuracy', mode='max', verbose=1, save_best_only=True)

  tf_transform_output = tft.TFTransformOutput(fn_args.transform_output)
  train_dataset = input_fn(fn_args.train_files,
                           tf_transform_output,
                           batch_size=BATCH_SIZE)
  eval_dataset = input_fn(fn_args.eval_files,
                          tf_transform_output,
                          batch_size=BATCH_SIZE)

  if fn_args.hyperparameters:
    hparams = kerastuner.HyperParameters.from_config(fn_args.hyperparameters)
  else:
    hparams = get_hyperparameters()


  print(f'hyperband parameters: {hparams.get_config()}')
  model = model_fn(hparams)

  model.fit(
    x = train_dataset,
    epochs= 5,
    steps_per_epoch = fn_args.train_steps,
    validation_data = eval_dataset,
    validation_steps = fn_args.eval_steps,
    callbacks = [tensorboard_callback, stop_early, model_chekpoint])

  signatures = {
      "serving_default":
      get_serve_tf_examples_fn(model,
                               tf_transform_output).get_concrete_function(
                                tf.TensorSpec(
                                  shape=[None],
                                  dtype=tf.string,
                                  name=FEATURE_KEY
                                )),
    }
  model.save(fn_args.serving_model_dir, save_format="tf", signatures=signatures)




Overwriting emotion_model_tuner.py


In [24]:
transform  = Transform(
    examples=example_gen.outputs['examples'],
    schema= schema_gen.outputs['schema'],
    module_file=os.path.abspath(TUNER_MODEL_MODULE_FILE)
)
interactive_content.run(transform)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('signature_function', 'signature_key'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('signature_function', 'signature_key'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


ExecutionResult(
    component_id: Transform
    execution_id: 6
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

In [25]:
train_uri = os.path.join(transform.outputs['transformed_examples'].get()[0].uri, "Split-train")
tfrecord_filenames = [os.path.join(train_uri, name)
                      for name in os.listdir(train_uri)]

dataset = tf.data.TFRecordDataset(tfrecord_filenames, compression_type="GZIP")

for tfrecord in dataset.take(1):
  serialized_example = tfrecord.numpy()
  example = tf.train.Example()
  example.ParseFromString(serialized_example)
  print(example)

features {
  feature {
    key: "label_xf"
    value {
      int64_list {
        value: 4
      }
    }
  }
  feature {
    key: "text_xf"
    value {
      bytes_list {
        value: "i gave up my internship with the dmrg and am feeling distraught"
      }
    }
  }
}



## Tuner

In [ ]:
tuner = Tuner(
    module_file = os.path.abspath(TUNER_MODEL_MODULE_FILE),
    examples = transform.outputs['transformed_examples'],
    transform_graph = transform.outputs['transform_graph'],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=30),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=10)
)

interactive_content.run(tuner)

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_xf (InputLayer)        [(None,)]                    0         []                            
                                                                                                  
 tokenizer (KerasLayer)      (None, None, None)           0         ['text_xf[0][0]']             
                                                                                                  
 packer (KerasLayer)         {'input_word_ids': (None,    0         ['tokenizer[0][0]']           
                             64),                                                                 
                              'input_mask': (None, 64),                                           
                              'input_type_ids': (None,                                      

Instructions for updating:
Use `tf.data.Dataset.map(tf.io.parse_example(...))` instead.


Search space summary
Default search space size: 2
learning_rate (Choice)
{'default': 0.001, 'conditions': [], 'values': [0.0001, 0.001, 0.01], 'ordered': True}
dropout (Choice)
{'default': 0.3, 'conditions': [], 'values': [0.0, 0.2, 0.3, 0.4], 'ordered': True}

Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
0.0001            |0.0001            |learning_rate
0                 |0                 |dropout
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
0                 |0                 |tuner/round

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_xf (InputLayer)        [(None,)]                    0         []                            
                                   

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: closure mismatch, requested ('self', 'step_function'), but source function had ()
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


## Trainer

In [ ]:
trainer = Trainer(
    module_file = os.path.abspath(TUNER_MODEL_MODULE_FILE),
    examples = transform.outputs['transformed_examples'],
    transform_graph = transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=["train"]),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"])
)

interactive_content(trainer)